In [1]:
import os
import json
from tqdm.auto import tqdm
from pathlib import Path

import pandas as pd

from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
# RUN ONCE: change working directory to project root
cwd = Path.cwd()

pwd = cwd.parent

os.chdir(pwd)
print(f"Changed working directory to {pwd}")

if Path.cwd() != pwd:
    raise RuntimeError(f"Failed to change working directory to {pwd}")

Changed working directory to /Users/jdk/projects/recruiting-reader


In [3]:
# create driver
from src.scraper.driver import make_driver
driver = make_driver(headless=True)

In [ ]:
from src.config import PORTAL_2025, PORTAL_2024
from src.scraper.link_extractor import scrape_portal_player_links


# urls_2024 = scrape_portal_player_links(driver, PORTAL_2024)
# print(f"Found {len(urls_2024)} player transfer portal links for 2024 class.")

urls_2025 = scrape_portal_player_links(driver, PORTAL_2025)
print(f"Found {len(urls_2025)} player transfer portal links for 2025 class.")

with open("data/urls_2025_transfers.json", "w") as f:
    json.dump(urls_2025, f)

# all_urls = list(dict.fromkeys(urls_2025 + urls_2024))
# print("total portal urls:", len(all_urls))

Clicking 'Load More':   0%|          | 0/200 [00:00<?, ?click/s]

Found 3010 player transfer portal links for 2025 class.


In [5]:
# urls_2025[2:]
# first two results erroneous cbssports.com links

In [6]:
## tests
# from src.scraper.player_scraper import scrape_player
# p = scrape_player(driver, 'https://247sports.com/player/emmanuel-pregnon-46140927/college-299861/')
# p

# from src.utils.tests import test_timeline
# events = test_timeline(driver, "https://247sports.com/player/howard-sampson-46129672/college-310950/")
# len(events)

# from src.config import DEBUG
# from src.scraper.player_scraper import scrape_player

# rows = []
# for u in tqdm(urls_2025[2:102]):
#     try:
#         d = scrape_player(driver, u)
#         rows.append(d)
#     except Exception as e:
#         print("FAIL", u, e)

# portal_df = pd.DataFrame(rows)
# portal_df.isna().sum()

In [4]:
with open("data/urls_2025_transfers.json", "r") as f:
    urls_2025 = json.load(f)

In [5]:
from src.storage.cache_new import run_scrape
# from src.config import CACHE_PATH

CACHE_PATH = "data/portal_2025_transfers_1218_run5.jsonl"

all_urls = list(dict.fromkeys(urls_2025[2:]))
scraped_count, cache_count = run_scrape(
    all_urls,
    out_path=CACHE_PATH,
    num_workers=8,        # try 4, 6, 8
    recycle_every=100     # try 50 if you see instability
)
print("scraped this run:", scraped_count, "already cached:", cache_count)


Scraping: 0player [00:00, ?player/s]

scraped this run: 0 already cached: 3008


In [18]:
from src.storage.cache_new import load_cache

test = load_cache(CACHE_PATH)
test_df = pd.DataFrame(test).T.set_index('id_247')
test_clean = {
    k: v
    for k, v in test.items()
    if (
        isinstance(v, dict)
        # and v.get("pos_247") is not None
        # and v.get("hs_city") is not None
        # and v.get("transfer_origin") is not None
        # and v.get("transfer_destination") is not None
    )
}
test_clean_df = pd.DataFrame(test_clean).T.set_index('id_247')
test_clean_df.to_csv(CACHE_PATH[:-5] + 'csv')

with open(CACHE_PATH, "w") as f:
    for v in test_clean.values():
            f.write(json.dumps(v) + "\n")

In [19]:
display(test_clean_df.isna().sum())
display(len(test_clean))

name                       0
pos_247                    0
hs_name                    5
hs_city                    0
hs_state                   0
transfer_rating            0
transfer_year              0
transfer_ovr_rank         88
transfer_pos_rank         35
transfer_stars             0
transfer_origin            5
transfer_destination      68
hs_class                   1
hs_rating_247           1012
hs_pos                     0
composite_rating        1121
composite_natl_rank     1123
composite_pos_rank      1123
source_hs_url              0
hs_stars                1012
source_player_url          0
transfer_status         2862
dtype: int64

3008